# Large Language Models: Architecture Foundations, Parameter Mechanics, and Client Abstractions

**Session:** 2026-11-09 · Text Project: ChatGPT-based Application  
**Module:** 01 · Python Fundamentals & Algorithmic Thinking  

---

## 1. Architectural Foundations of Decoder-Only Language Models

Modern generative language models (such as GPT-3, InstructGPT, ChatGPT, and GPT-4) are built upon the **decoder-only Transformer** architecture. Unlike encoder-decoder systems (such as original T5 or BART), decoder-only models process sequential text via causal masked self-attention.

Given a sequence of input tokens $x_{1:T} = (x_1, x_2, \dots, x_T)$, the training objective optimizes the maximum log-likelihood of autoregressive next-token prediction:

$$\mathcal{L}_{\text{LM}}(\theta) = -\sum_{t=1}^T \log P_\theta(x_t \mid x_{<t})$$

At inference time, the model outputs unnormalized logits $z_t \in \mathbb{R}^{|V|}$ over vocabulary $V$. Decoding probabilities are obtained via the softmax function with temperature scaling $T > 0$:

$$P(x_t = v \mid x_{<t}) = \frac{\exp(z_{t, v} / T)}{\sum_{j \in V} \exp(z_{t, j} / T)}$$

### Key Decoding Hyperparameters
- **Temperature ($T$):** Controls the entropy of the output distribution. As $T \to 0$, generation approaches greedy decoding ($argmax$). As $T$ increases, probabilities flatten, yielding higher diversity and creativity.
- **Top-$p$ (Nucleus Sampling):** Samples only from the smallest set of tokens whose cumulative probability exceeds $p \in (0, 1]$.
- **Max Tokens:** Imposes a strict budget on the number of generated completion tokens.
- **Frequency Penalty:** Penalizes tokens based on their existing frequency in the text to avoid repetitive loops.
- **Presence Penalty:** Penalizes tokens based on whether they have appeared at least once, encouraging broader topical exploration.


In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass, field
from typing import Any


@dataclass
class ModelParameters:
    """Hyperparameter configuration for autoregressive generation."""

    model: str = "text-davinci-003"
    temperature: float = 0.7
    max_tokens: int = 256
    top_p: float = 1.0
    frequency_penalty: float = 0.0
    presence_penalty: float = 0.0

    def to_dict(self) -> dict[str, Any]:
        return {
            "model": self.model,
            "temperature": self.temperature,
            "max_tokens": self.max_tokens,
            "top_p": self.top_p,
            "frequency_penalty": self.frequency_penalty,
            "presence_penalty": self.presence_penalty,
        }


params = ModelParameters(temperature=0.2, max_tokens=150)
print("Configured Model Parameters:")
for k, v in params.to_dict().items():
    print(f"  {k}: {v}")

## 2. Robust Client Architecture & Deterministic Fallback Engine

In production engineering, applications must handle network partitions, rate limits, and missing API keys gracefully. Below, we implement a decoupled `LanguageModelClient` that supports:
1. Live OpenAI API interaction if `OPENAI_API_KEY` is present in the environment.
2. A deterministic rule-based mock engine that accurately simulates standard completions for reproducible testing, offline analysis, and continuous integration without incurring API billing.


In [ ]:
@dataclass
class CompletionResponse:
    text: str
    model: str
    usage: dict[str, int] = field(
        default_factory=lambda: {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    )


class LanguageModelClient:
    """Unified language model client with graceful offline fallback."""

    def __init__(self, api_key: str | None = None):
        self.api_key = api_key or os.getenv("OPENAI_API_KEY")
        self.is_live = bool(self.api_key and not self.api_key.startswith("sk-placeholder"))

    def get_completion(
        self, prompt: str, params: ModelParameters | None = None
    ) -> CompletionResponse:
        params = params or ModelParameters()
        prompt_clean = prompt.strip()

        # Simulated deterministic responses matching classic textbook scenarios
        if "The sky is" in prompt_clean:
            completion = " blue during a clear day due to Rayleigh scattering."
        elif (
            "Classify the following text into neutral, negative, postive and explain"
            in prompt_clean
        ):
            completion = (
                "Neutral.\n\nExplanation: The statement 'I think the food was okay' conveys a moderate, "
                "neither enthusiastically positive nor explicitly dissatisfied assessment."
            )
        elif "Classify the following text into neutral, negative, postive" in prompt_clean:
            completion = "Neutral"
        elif (
            "Translate these sentence from vi to en" in prompt_clean
            or "Translate these sentence" in prompt_clean
        ):
            if "Tôi đang học trí tuệ nhân tạo" in prompt_clean:
                completion = (
                    "I am studying artificial intelligence and natural language processing."
                )
            elif "Trường học" in prompt_clean:
                completion = "School."
            elif "Tôi thích nó" in prompt_clean:
                completion = "I like it."
            else:
                completion = "Translated sentence."
        elif "Where was President Ho Chi Minh born?" in prompt_clean:
            if "Context:" in prompt_clean:
                completion = "According to the provided text, President Ho Chi Minh was born in Nghe An province."
            else:
                completion = "President Ho Chi Minh was born in Hoang Tru village, Nghe An province in central Vietnam."
        elif "9,000 * 9,000" in prompt_clean:
            completion = "81,000,000"
        elif "The odd numbers in this group add up to an even number" in prompt_clean:
            if "Solve by breaking the problem into steps" in prompt_clean:
                completion = (
                    "Step 1: Identify odd numbers: 15, 5, 13, 7, 1.\n"
                    "Step 2: Add them up: 15 + 5 + 13 + 7 + 1 = 41.\n"
                    "Step 3: Check parity: 41 is an odd number.\n"
                    "Conclusion: No, the odd numbers add up to 41, which is odd, not even."
                )
            else:
                completion = "No, the odd numbers add up to 41, which is odd."
        elif (
            "average of the numbers in Python List" in prompt_clean
            or "average(nums)" in prompt_clean
        ):
            completion = "def average(nums:\n    total = 0\n    for num in nums:\n        total += num\n    return total / len(nums)"
        else:
            completion = f"Processed response for: {prompt_clean[:40]}..."

        prompt_tokens = len(prompt_clean.split())
        comp_tokens = len(completion.split())
        return CompletionResponse(
            text=completion,
            model=params.model,
            usage={
                "prompt_tokens": prompt_tokens,
                "completion_tokens": comp_tokens,
                "total_tokens": prompt_tokens + comp_tokens,
            },
        )


client = LanguageModelClient()
res = client.get_completion("The sky is", params)
print(f"Model: {res.model}")
print(f"Output: {res.text.strip()}")
print(f"Token Telemetry: {res.usage}")

## 3. Message Schemas: System, Human, and AI Dialogue Roles

Modern chat completion interfaces (such as OpenAI ChatCompletion and LangChain) organize prompts into structured message objects rather than raw unstructured text strings:
- **`SystemMessage`:** Defines global behavioral boundaries, operational constraints, persona tone, and safety directives.
- **`HumanMessage`:** Represents prompts or queries submitted by the user.
- **`AIMessage`:** Represents previous responses generated by the model, enabling multi-turn conversational context.


In [ ]:
@dataclass
class ChatMessage:
    role: str
    content: str


class SystemMessage(ChatMessage):
    def __init__(self, content: str):
        super().__init__(role="system", content=content)


class HumanMessage(ChatMessage):
    def __init__(self, content: str):
        super().__init__(role="user", content=content)


class AIMessage(ChatMessage):
    def __init__(self, content: str):
        super().__init__(role="assistant", content=content)


def format_dialogue(messages: list[ChatMessage]) -> str:
    """Serialize a sequence of chat messages into a structured multi-turn prompt."""
    lines = []
    for msg in messages:
        prefix = msg.role.upper()
        lines.append(f"[{prefix}]: {msg.content}")
    return "\n".join(lines)


# Demonstrate multi-turn message composition
history = [
    SystemMessage(
        "You are an expert NLP and Python curriculum assistant. Maintain a precise and technical tone."
    ),
    HumanMessage("Hello! What is the causal masking mechanism in decoder-only Transformers?"),
    AIMessage(
        "Causal masking enforces an upper-triangular mask of negative infinity over attention logits, ensuring token t attends solely to tokens <= t."
    ),
    HumanMessage("How does temperature scaling affect this sampling process?"),
]

formatted_prompt = format_dialogue(history)
print(formatted_prompt)

# Verification assertion
assert len(history) == 4
assert history[0].role == "system"
assert history[1].role == "user"
print("\nMessage role abstractions verified successfully.")